In [ ]:
import torch
import torch.nn as nn

class SpaMoBaseline(nn.Module):
    def __init__(self, llm_hidden_size=768):
        super().__init__()
        
        self.spatial_projection = nn.Linear(in_features=768, out_features=llm_hidden_size)
        self.motion_projection  = nn.Linear(in_features=1024, out_features=llm_hidden_size)
        
        self.conv1d_sequence = nn.Sequential(
            nn.Conv1d(llm_hidden_size, llm_hidden_size, kernel_size=5, padding=2),
            nn.MaxPool1d(kernel_size=2, stride=2, padding=0),
            nn.ReLU(),
            nn.Conv1d(llm_hidden_size, llm_hidden_size, kernel_size=5, padding=2),
            nn.MaxPool1d(kernel_size=2, stride=2, padding=0),
            nn.ReLU()
        )
        
        self.encoder_decoder_adapter = nn.Sequential(
            nn.Linear(llm_hidden_size, 4 * llm_hidden_size),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(4 * llm_hidden_size, llm_hidden_size),
            nn.Dropout(0.1)
        )

    def forward(self, spatial_feats, motion_feats):
        spatial_projection        = self.spatial_projection(spatial_feats)
        motion_projection         = self.motion_projection(motion_feats)   
        representation_combination = spatial_projection + motion_projection   
        representation_combination = self.conv1d_sequence(representation_combination)
        representation_combination = self.encoder_decoder_adapter(representation_combination)
        return None